<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main/sankey2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Import libaries

In [69]:
import requests
import pandas as pd
import plotly.graph_objects as go
import duckdb
import string
import plotly.colors as pc


In [70]:
# Airtable credentials
api_key = 'patwjsizhgQyQkZkT.f9e8b1595df5b527d0d01d3a45af0dfa77eab63707e18398ad62f1f3818a9ce9'
base_id = 'apprKfEKZ2Ju74g9w'

In [71]:
# henter data fra Airtable via API og gemmer det i en pandas.DataFrame
def fetch_airtable_data(table_id):
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {api_key}"}
    records = []
    params = {}

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data: {response.text}")
        data = response.json()
        records.extend([record["fields"] | {"id": record["id"]} for record in data["records"]])
        if "offset" in data:
            params["offset"] = data["offset"]
        else:
            break

    return pd.DataFrame(records)

In [72]:
# Fetch data
tabel0 = fetch_airtable_data("tblVarbVYd96JUE6f")  # Target Group
tabel1 = fetch_airtable_data("tbl7OYOXduME11uh7")  # Targets
tabel2 = fetch_airtable_data("tblTRyuT48bBN24QG")   # Slut-land uses

In [73]:
# For t0 (Target Group)
t0 = tabel0.copy()
if 'Targets' in t0.columns:
    t0 = t0.explode('Targets')
t0 = t0.rename(columns={
    'Target Group': 'target_group_name',
    'Targets': 'target_id',           # omdøber eksploderede Targets til target_id
    'id': 'target_group_id'
})

# For t1 (Targets)
t1 = tabel1.copy()
if 'Land uses' in t1.columns:
    t1 = t1.explode('Land uses')
t1 = t1.rename(columns={
    'Target name': 'target_name',
    'id': 'target_id',        # omdøb id
    'Land uses': 'land_use_id'
})

# For t2 (Land Uses)
t2 = tabel2.copy()
t2 = t2.rename(columns={
    'Name': 'land_use_name',
    'id': 'land_use_id'       # omdøb id
})


In [74]:
# Opret forbindelse
con = duckdb.connect()

# Registrer pandas DataFrames som tabeller i DuckDB
con.register('tabel0', t0)
con.register('tabel1_exp', t1)
con.register('tabel2_exp', t2)


In [76]:
result = con.sql("""
SELECT
  tg.target_group_name,
  t.target_name,
  lu.land_use_name
FROM t0 tg
JOIN t1 t ON tg.target_id = t.target_id
JOIN t2 lu ON t.land_use_id = lu.land_use_id
LIMIT 5
""").df()

In [78]:
def forkort_label(label, max_len=40):
    if isinstance(label, str) and len(label) > max_len:
        return label[:max_len] + '…'
    return label

# Anvend på kolonner i join-resultat (fx result)
result['target_group_name_short'] = result['target_group_name'].apply(forkort_label)
result['land_use_name_short'] = result['land_use_name'].apply(forkort_label)

In [79]:
# 1. Alle labels
all_labels = pd.concat([
    result['target_group_name_short'],
    result['target_name'],
    result['land_use_name_short']
]).unique().tolist()

# 2. Map label til indeks
label_to_index = {label: i for i, label in enumerate(all_labels)}

# 3. Kilde (source) og mål (target) for links
source = result['target_group_name_short'].map(label_to_index)
target = result['target_name'].map(label_to_index)
value = [1] * len(result)  # Vægt 1 pr række

# 4. Andet led links
source2 = result['target_name'].map(label_to_index)
target2 = result['land_use_name_short'].map(label_to_index)
value2 = [1] * len(result)

# 5. Saml alle links
source_all = pd.concat([source, source2], ignore_index=True)
target_all = pd.concat([target, target2], ignore_index=True)
value_all = pd.concat([pd.Series(value), pd.Series(value2)], ignore_index=True)

In [83]:
# 1. Lav labels direkte ud fra dine 3 kolonner i result
def preprocess_label(label, max_len=40, wrap_len=60):
    if not isinstance(label, str):
        return label
    if len(label) > max_len:
        label = label[:max_len] + '…'
    return '\n'.join([label[i:i+wrap_len] for i in range(0, len(label), wrap_len)])

# Anvend direkte uden at ændre originalt DataFrame
tg_labels = result['target_group_name'].apply(lambda x: preprocess_label(x))
t_labels  = result['target_name'].apply(lambda x: preprocess_label(x))
lu_labels = result['land_use_name'].apply(lambda x: preprocess_label(x))

# 2. Saml unikke labels i rækkefølge
all_labels = pd.concat([tg_labels, t_labels, lu_labels]).unique().tolist()

# 3. Map labels til node-indeks
label_to_index = {label: i for i, label in enumerate(all_labels)}

# 4. Opret source/target/value-lister
source = tg_labels.map(label_to_index)
target = t_labels.map(label_to_index)
value = [1] * len(result)

source2 = t_labels.map(label_to_index)
target2 = lu_labels.map(label_to_index)
value2 = [1] * len(result)

# Saml alt
source_all = pd.concat([source, source2], ignore_index=True)
target_all = pd.concat([target, target2], ignore_index=True)
value_all = pd.Series(value + value2)

# 5. Farver
node_colors = pc.qualitative.Plotly
color_map = {label: node_colors[i % len(node_colors)] for i, label in enumerate(all_labels)}
node_colors_list = [color_map[label] for label in all_labels]

def lighten(hex_color, factor=0.5):
    from plotly.colors import hex_to_rgb
    r, g, b = hex_to_rgb(hex_color)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'rgb({r},{g},{b})'

link_colors = [lighten(node_colors_list[src], factor=0.8) for src in source_all]

# 6. Sankey
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=60,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=all_labels,
        color=node_colors_list
    ),
    link=dict(
        source=source_all,
        target=target_all,
        value=value_all,
        color=link_colors
    )
)])

fig.update_layout(
    title_text="Target Group → Target → Land Use",
    font_size=12,
    height=1200
)

fig.show()


#gammel

In [ ]:
#fig.update_layout(title_text="Policy Source → Target Group → Target", font_size=12, height=2000)
#fig.write_html("sankey_diagram.html")

#from google.colab import files
#files.download("sankey_diagram.html")